### Install Necessary Libraries

In [1]:
!pip install colbert-ai[faiss-gpu,torch]
!pip install python-terrier
!pip install ir-datasets
!pip install numpy pandas scipy scikit-learn
!pip install transformers
!pip install sentence-transformers
!pip install ranx 
!pip install pytrec_eval
!pip install tqdm
!pip install datasets 
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.4/887.4 MB 287.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 MB 287.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.0/21.0 MB 291.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 485.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.1/557.1 MB 293.7 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: nvidia-cuda-runtime-cu11
    Found existing installation: nvidia-cuda-runtime-cu11 11.8.89
    Uninstalling nvidia-cuda-runtime-cu11-11.8.89:
      Successfully uninstalled nvidia-cuda-runtime-cu11-11.8.890m [nvidia-cuda-runtime-cu11]
  Attempting uninstall: nvidia-cuda-nvrtc-cu11━━ 0/5 [nvidia-cuda-runtime-cu11]
    Found existing installation: nvidia-cuda-nvrtc-cu11 11.8.89nvidia-cuda-runtime-cu11]
    Uninstalling nvidia-cuda-nvrtc-cu11-11.8.89: 0/5 [nvidia-cuda-runtime-cu11]
      Successfully uninstalled nvidia-cuda-nvrtc-cu11-11.8.8

### Import Necessary libraries

In [2]:
import os
import torch
import numpy as np
import ir_datasets
import pandas as pd
import pyterrier as pt
from colbert import Searcher
from scipy.sparse import csr_matrix
from colbert.infra import Run, RunConfig
from transformers import AutoTokenizer, AutoModelForMaskedLM

/opt/miniconda3/envs/colbertv2/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Initialize PyTerrier
if not pt.java.started():
    pt.java.init()

Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


### Load Dataset

In [16]:
# Load the SARA dataset from ir_datasets
dataset = ir_datasets.load('sara')

# Extract queries and relevance judgments into lists
topics_list = list(dataset.queries_iter())
qrels_list = list(dataset.qrels_iter())

# Collect all document texts
docs = list(dataset.docs_iter())
doc_texts = {doc.doc_id: doc.text for doc in docs}

# Convert queries to a PyTerrier-compatible DataFrame
topics = pd.DataFrame([
    {'qid': topic.query_id, 'query': topic.text} 
    for topic in topics_list
])

# Convert qrels (relevance judgments) to a PyTerrier-compatible DataFrame
qrels = pd.DataFrame([
    {'qid': qrel.query_id, 'docno': str(qrel.doc_id), 'label': int(qrel.relevance)}
    for qrel in qrels_list
])

# Print dataset statistics
print(f"No. of Queries: {len(topics)},  No. of Documents: {len(docs)}, Total Qrels: {len(qrels)}")

No. of Queries: 150,  No. of Documents: 1702, Total Qrels: 34413


### Setting Up Document ID Mapping

In [17]:
# Load the document ID mapping
index_name = 'colbert_sara_index'
mapping_path = f"./Index/{index_name}_doc_mapping.txt"

print(f"Document ID mapping from: {mapping_path} loaded ")
if not os.path.exists(mapping_path):
    raise FileNotFoundError(f"Mapping file not found: {mapping_path}")

# Load mapping: ColBERT index -> SARA doc ID
idx_to_doc_id = {}
with open(mapping_path, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 2:
            idx, doc_id = parts
            idx_to_doc_id[int(idx)] = doc_id

print(f"Mapping for {len(idx_to_doc_id)} documents loaded \n")
print(f"Sample mappings: {dict(list(idx_to_doc_id.items())[:3])}")

Document ID mapping from: ./Index/colbert_sara_index_doc_mapping.txt loaded 
Mapping for 1702 documents loaded 

Sample mappings: {0: '114715', 1: '229405', 2: '232795'}


### Load ColBERT Retrieval Model

In [18]:
# Load the ColBERT index
print("Loading ColBERT searcher")

with Run().context(RunConfig(experiment='sara_experiment')):
    searcher = Searcher(index=index_name, checkpoint="colbert-ir/colbertv2.0")
    print("Searcher loaded successfully")

Loading ColBERT searcher
[Sep 08, 14:56:12] #> Loading codec...
[Sep 08, 14:56:12] #> Loading IVF...
[Sep 08, 14:56:12] #> Loading doclens...


/opt/miniconda3/envs/colbertv2/lib/python3.9/site-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
100%|██████████| 1/1 [00:00<00:00, 1285.02it/s]

[Sep 08, 14:56:12] #> Loading codes and residuals...



100%|██████████| 1/1 [00:00<00:00, 69.79it/s]

Searcher loaded successfully


### Load Splade Retrieval Model

In [19]:
# Load SPLADEv2 model and tokenizer
splade_model_name = "naver/splade-cocondenser-ensembledistil"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
splade_tokenizer = AutoTokenizer.from_pretrained(splade_model_name)
splade_model = AutoModelForMaskedLM.from_pretrained(splade_model_name).to(device)

# Set to evaluation mode
splade_model.eval()

BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwi

### Document Encoding and Retrieval Functions 
##### Top K documents retrieved by ColBERT are re-ranked by SPLADE 


In [20]:
def encode_splade(text, tokenizer, model, max_length=512):

    # Tokenize input text
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=max_length).to(device)
    
    # Generate SPLADE representation without gradient computation
    with torch.no_grad():
        
        logits = model(**inputs).logits
        # SPLADE log(1 + relu(x)) sparse encoding
        sparse_rep = torch.log(1 + torch.relu(logits)) * inputs.attention_mask.unsqueeze(-1)
        sparse_rep = torch.max(sparse_rep, dim=1)[0]  # max pooling over sequence length
        
    return sparse_rep.cpu().numpy()

def compute_splade_similarity(query_rep, doc_rep):
    "Compute similarity between SPLADE representations"
    
    # Convert to sparse matrices for efficient computation
    query_sparse = csr_matrix(query_rep)
    doc_sparse = csr_matrix(doc_rep)
    
    # Compute dot product
    similarity = query_sparse.dot(doc_sparse.T).toarray()[0, 0]
    return float(similarity)

def retrieve_and_rerank(topics_df, k_colbert=100, k_rerank=100):
    "Retrieve with ColBERT and re-rank top results with SPLADE v2"
    final_results = []
    
    print(f"Running ColBERT + SPLADE v2 pipeline for {len(topics_df)} queries...")
    
    for i, (_, row) in enumerate(topics_df.iterrows()):
        if i % 10 == 0:
            print(f"Processing query {i+1}/{len(topics_df)}")
            
        qid = row['qid']
        query = row['query']
        
        try:
            # ColBERT retrieval
            pids, ranks, scores = searcher.search(query, k=k_colbert)
            
            # Get top k results for re-ranking
            colbert_results = []
            for rank, (pid, score) in enumerate(zip(pids[:k_rerank], scores[:k_rerank])):
                if pid in idx_to_doc_id:
                    original_doc_id = idx_to_doc_id[pid]
                    if original_doc_id in doc_texts:
                        colbert_results.append({
                            'doc_id': original_doc_id,
                            'text': doc_texts[original_doc_id],
                            'colbert_score': float(score),
                            'colbert_rank': rank
                        })
            
            if not colbert_results:
                print(f"No valid results for query {qid}")
                continue
            
            # SPLADE v2 re-ranking
            print(f"  Re-ranking top {len(colbert_results)} results with SPLADE v2...")
            
            # Encode query with SPLADE
            query_rep = encode_splade(query, splade_tokenizer, splade_model)
            
            # Encode documents and compute similarities
            splade_results = []
            for result in colbert_results:
                doc_rep = encode_splade(result['text'], splade_tokenizer, splade_model)
                splade_score = compute_splade_similarity(query_rep, doc_rep)
                
                splade_results.append({
                    'doc_id': result['doc_id'],
                    'splade_score': splade_score,
                    'colbert_score': result['colbert_score'],  # Keep for reference
                    'colbert_rank': result['colbert_rank']
                })
            
            # Sort by SPLADE score
            splade_results.sort(key=lambda x: x['splade_score'], reverse=True)
            
            # Add to final results with SPLADE ranking
            for rank, result in enumerate(splade_results):
                final_results.append({
                    'qid': qid,
                    'docno': str(result['doc_id']),
                    'score': result['splade_score'],
                    'colbert_score': result['colbert_score'],
                    'splade_score': result['splade_score'],
                    'colbert_rank': result['colbert_rank'],
                    'rank': rank  # New SPLADE ranking
                })
            
        except Exception as e:
            print(f"Error processing query {qid}: {e}")
            continue
    
    # Convert to DataFrame
    results_df = pd.DataFrame(final_results)
        
    return results_df

In [23]:
# Run the pipeline
scores = retrieve_and_rerank(topics, k_colbert=100, k_rerank=100)
print(f"Retrieved and re-ranked {len(scores)} results")

if len(scores) == 0:
    print("ERROR: No results retrieved! Check your pipeline.")
    exit(1)

Running ColBERT + SPLADE v2 pipeline for 150 queries...
Processing query 1/150
  Re-ranking top 100 results with SPLADE v2...
  Re-ranking top 100 results with SPLADE v2...
  Re-ranking top 100 results with SPLADE v2...
  Re-ranking top 100 results with SPLADE v2...
  Re-ranking top 100 results with SPLADE v2...
  Re-ranking top 100 results with SPLADE v2...
  Re-ranking top 100 results with SPLADE v2...
  Re-ranking top 100 results with SPLADE v2...
  Re-ranking top 100 results with SPLADE v2...
  Re-ranking top 100 results with SPLADE v2...
Processing query 11/150
  Re-ranking top 100 results with SPLADE v2...
  Re-ranking top 100 results with SPLADE v2...
  Re-ranking top 100 results with SPLADE v2...
  Re-ranking top 100 results with SPLADE v2...
  Re-ranking top 100 results with SPLADE v2...
  Re-ranking top 100 results with SPLADE v2...
  Re-ranking top 100 results with SPLADE v2...
  Re-ranking top 100 results with SPLADE v2...
  Re-ranking top 100 results with SPLADE v2...
  Re

In [ ]:
scores.to_csv("./retrieval_scores/colbert+splade_scores.csv")

In [25]:
print(scores.shape)
scores.head()

(15000, 7)


,qid,docno,score,colbert_score,splade_score,colbert_rank,rank
0,1,120514,6.148230,10.804688,6.148230,3,0
1,1,175162,6.029339,8.593750,6.029339,36,1
2,1,6178,5.775954,8.117188,5.775954,49,2
3,1,177853,5.539324,9.828125,5.539324,10,3
4,1,176650,5.378632,9.703125,5.378632,12,4


### Evaluation

In [26]:
# Evaluate with PyTerrier
scores['docno'] = scores['docno'].astype(str)
qrels['docno'] = qrels['docno'].astype(str)

metrics = ['ndcg_cut_10', 'map', 'P_10', 'recall_10']
eval_results = pt.Evaluate(scores, qrels, metrics=metrics)

print("\nEvaluation")


Evaluation


In [28]:
results = pd.DataFrame(eval_results, index=[0])
results = results.round(4)

# Add model column
results.insert(0, "Model", "ColBERT+SPLADE")

In [29]:
results

,Model,ndcg_cut_10,map,P_10,recall_10
0,ColBERT+SPLADE,0.1513,0.0889,0.1347,0.1227
